# Top-32: saved score index 22, prune before LLM vs after block index 2
GPU 0: score22_prune_input; GPU 1: score22_prune_after2. Both use the SAME global Top-32 selections, full 16,817 test samples, greedy FP32 batch 64/GPU. Index 22 means layer 23; gather after index 2 means after the COMPLETE third block. QA-quality benchmark, not a latency/FLOPs measurement.

Push implemented code to smolvlm-attention before running. Enable Internet and two GPUs. Attach the existing cache and extracted TEST PROBABILITY attention export. Teacher train/validation log-targets are incompatible.


In [ ]:
# 1. Run before importing Torch/CUDA. Do not update source while workers are running.
from pathlib import Path
import os, subprocess, sys
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
SOURCE = Path('/kaggle/working/Log_Metrics')
OUTPUT = Path('/kaggle/working/smolvlm_top32_score22_input_vs_after2_outputs')
RUN_MODE = 'auto'
ATTENTION_INPUT = ''
CACHE_INPUT = ''
PREPARED_DIR = ''
# RUN_MODE: auto=create/resume/reuse; measure=new only; resume=existing output required.
# ATTENTION_INPUT: required extracted test probability export folder.
# CACHE_INPUT: blank uses cache resolver, or supply Input folder with memmap/manifest/index.
# PREPARED_DIR: blank downloads pinned public prepared data.


In [ ]:
# 2. Check source origin/branch; preserve tracked local changes.
if not SOURCE.exists():
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch','smolvlm-attention',
                    'https://github.com/huyvanzzz/Log_Metrics.git',str(SOURCE)],check=True)
else:
    def git(*args):
        return subprocess.check_output(['git','-C',str(SOURCE),*args],text=True).strip()
    if git('branch','--show-current') != 'smolvlm-attention' or git('remote','get-url','origin').rstrip('/') != 'https://github.com/huyvanzzz/Log_Metrics.git':
        raise RuntimeError('Source branch/origin mismatch; choose a new SOURCE folder.')
    if git('status','--porcelain','--untracked-files=no'):
        raise RuntimeError('Preserve tracked local edits before pulling; no reset was performed.')
    subprocess.run(['git','-C',str(SOURCE),'pull','--ff-only','origin','smolvlm-attention'],check=True)
os.chdir(SOURCE)
if not (SOURCE/'smolvlm_topk_deferred/__main__.py').is_file():
    raise RuntimeError('New benchmark code is missing on GitHub. Push the implemented branch first.')
subprocess.run([sys.executable,'-m','pip','install','--only-binary=tokenizers',
                '-r',str(SOURCE/'smolvlm_topk_benchmark/requirements.txt')],check=True)
import shutil
if shutil.which('java') is None:
    if shutil.which('apt-get') is None:
        raise RuntimeError('Install Java 8+ for METEOR, then rerun this cell.')
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jre-headless'],check=True)
print('Source commit:', subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
# 3. Validate dependencies and both CUDA devices. CLI runs in fresh subprocesses.
import torch, transformers, tokenizers
assert transformers.__version__ == '4.46.3'
assert tokenizers.__version__ == '0.20.4'
assert torch.cuda.device_count() >= 2, 'Enable two CUDA GPUs; no single-GPU fallback.'
for i in range(2):
    print(i, torch.cuda.get_device_name(i))

def run_backend(command, *args):
    subprocess.run([sys.executable,'-m','smolvlm_topk_deferred',command,
                    '--output',str(OUTPUT),*map(str,args)],cwd=SOURCE,check=True)


In [ ]:
# 4. Validate/alignment, prepare ONE shared selection, metric preflight.
# Cache is read directly; public HF downloads need no token or publishing credentials.
if not ATTENTION_INPUT:
    raise ValueError('Fill ATTENTION_INPUT with the complete, extracted base test probability export.')
options = ['--mode',RUN_MODE,'--attention-input',ATTENTION_INPUT,'--cache-input',CACHE_INPUT]
if PREPARED_DIR:
    options += ['--prepared-dir',PREPARED_DIR]
run_backend('prepare',*options)


In [ ]:
# 5. Numerical preflight and longest-64 stress on EACH GPU.
# Both must pass before full generation starts. Worker logs stream by GPU.
# OOM/error stops peer, keeps commits; no batch/precision fallback.
# Rerun this cell for partial output with identical code/dependencies/identity.
run_backend('run')


In [ ]:
# 6. Score all seven metrics and show delta after2 minus input.
run_backend('score')
import json
summary = json.loads((OUTPUT/'summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary,ensure_ascii=False,indent=2))


In [ ]:
# 7. Bundle small results only, without checkpoint/cache/source attention.
run_backend('bundle')
from IPython.display import FileLink, display
archive = OUTPUT/'top32_score22_input_vs_after2_results.zip'
display(FileLink(str(archive)))


## Resume and artifacts
- Same session: preserve OUTPUT and use RUN_MODE auto/resume, then prepare and run. Do not change source/dependencies mid-run.
- Across sessions: restore the FULL partial OUTPUT tree, not just the result ZIP. Resolve the same inputs again and run prepare to validate identity/update operational paths. No automatic partial upload or restoration.
- Final ZIP contains portable results, not complete resume state. Keep source checkpoint/cache/attention separately.
- CPU tests do not validate batch 64/GPU or VRAM. Real GPU stress is mandatory and may fail with OOM.
